# 05. 데이터 전처리 v2 — Log Base 정규화 (실험 3, 기각됨)

## 실험 개요

이 노트북은 실험 3의 전처리 단계다. 04 노트북에서 이론적으로 더 우수하다고 판단한 **Log Base 정규화** `log(P_t / P_0)`를 실제 데이터에 적용해 패턴 이미지를 생성한다.

v1과의 핵심 차이점:
- **정규화**: MinMaxScaler → `log(P_t / P_0)` (MinMaxScaler 없이 로그 값 그대로)
- **Anchor-Positive 쌍 오프셋**: +1/+2/+3주 → +6/+12주 (더 멀리 떨어진 시간 패턴을 유사로 정의)
- **저장 경로**: `data/` → `data_v2/`

## 결과 (07 노트북에서 최종 판정)

이 전처리로 학습한 모델(실험 3)은 Val Loss 0.138237로 실험 1(0.005174) 대비 **27배 높다**. 시각적 비교에서도 유사하지 않은 패턴을 반환해 **기각**되었다.

## 1. 환경 설정

필수 라이브러리를 설치하고 불러온다. v1과 동일한 의존성이지만 정규화 방법만 Log Base로 교체한다.

yfinance, mplfinance 등 필수 라이브러리를 설치하고 불러온다.

In [ ]:
# 필수 패키지 설치
!pip install yfinance mplfinance -q

import yfinance as yf
import numpy as np
import pandas as pd
import os
import gc
import time
import warnings
from datetime import datetime
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler

# 시각화
import matplotlib.pyplot as plt
import mplfinance as mpf
from PIL import Image, ImageEnhance

# Google Colab
from google.colab import drive

warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")

## 2. 경로 설정

Google Drive를 마운트하고 v2 전용 경로를 설정한다.

전처리는 Colab 로컬(`/content/data_v2`)에서 빠르게 처리하고, 최종 결과만 Google Drive(`data_v2/`)에 백업한다. v1 경로(`data/`)와 별도로 관리해 두 실험이 충돌하지 않도록 한다.

---

## 3. 종목 유니버스 정의

v1과 동일한 172개 종목 리스트를 재정의한다. 전처리 방법만 다르고 데이터 출처는 동일하다.

In [ ]:
# Google Drive 마운트
drive.mount('/content/drive')

# 경로 설정 (v2 버전!)
GDRIVE_BASE = '/content/drive/MyDrive/Patron'
COLAB_STORAGE = '/content/data_v2'  # 코랩 저장소 (임시, 빠름!)

# 전처리는 코랩 저장소에서 (빠른 I/O)
RAW_PATH = f'{COLAB_STORAGE}/raw'
PROCESSED_PATH = f'{COLAB_STORAGE}/processed_v2'
IMAGE_DIR = f'{COLAB_STORAGE}/images_v2'

# 최종 결과는 구글 드라이브에 백업 (영구 보관)
GDRIVE_DATA_V2 = f'{GDRIVE_BASE}/data_v2'

# 디렉토리 생성
os.makedirs(RAW_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(GDRIVE_DATA_V2, exist_ok=True)

print("="*60)
print("📁 경로 설정 (v2 버전)")
print("="*60)
print(f"✅ 코랩 저장소 Raw: {RAW_PATH}")
print(f"✅ 코랩 저장소 Processed: {PROCESSED_PATH}")
print(f"✅ 코랩 저장소 Images: {IMAGE_DIR}")
print(f"✅ 구글 드라이브 백업: {GDRIVE_DATA_V2}")
print(f"\n💡 코랩 저장소(/content)에서 처리 → 구글 드라이브 백업")
print("="*60)

## 4. 주봉 데이터 수집

172개 종목의 주봉 OHLC 데이터를 다운로드한다. v1과 동일한 방식으로 수집하되, Colab 로컬(`/content/data_v2/raw`)에 저장한다.

---

## 5. 섹터 정보 수집

172개 종목의 섹터·산업군 정보를 수집한다.

In [ ]:
# NASDAQ 100
nasdaq_100 = [
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'AVGO', 'GOOG', 'META',
    'TSLA', 'PLTR', 'NFLX', 'AMD', 'ASML', 'COST', 'CSCO', 'AZN',
    'MU', 'TMUS', 'SHOP', 'APP', 'PEP', 'LRCX', 'LIN', 'QCOM',
    'PDD', 'INTC', 'ISRG', 'INTU', 'AMAT', 'ARM', 'BKNG', 'AMGN',
    'KLAC', 'PANW', 'GILD', 'TXN', 'ADBE', 'CRWD', 'HON', 'MELI',
    'CEG', 'ADI', 'VRTX', 'DASH', 'ADP', 'CMCSA', 'CDNS', 'SBUX',
    'SNPS', 'MRVL', 'ORLY', 'ABNB', 'MSTR', 'MDLZ', 'CTAS', 'MAR',
    'TRI', 'REGN', 'CSX', 'FTNT', 'MNST', 'PYPL', 'AEP', 'ADSK',
    'WDAY', 'AXON', 'DDOG', 'WBD', 'NXPI', 'ZS', 'ROST', 'PCAR',
    'IDXX', 'EA', 'XEL', 'ROP', 'BKR', 'TTWO', 'FAST', 'EXC',
    'TEAM', 'PAYX', 'CPRT', 'FANG', 'CCEP', 'KDP', 'CTSH', 'GEHC',
    'MCHP', 'VRSK', 'CHTR', 'ODFL', 'KHC', 'CSGP', 'TTD', 'DXCM',
    'BIIB', 'CDW', 'ON', 'LULU', 'GFS'
]

# S&P 100
sp_100 = [
    'NVDA', 'AAPL', 'MSFT', 'GOOGL', 'GOOG', 'AMZN', 'AVGO', 'META',
    'TSLA', 'BRK-B', 'JPM', 'LLY', 'WMT', 'ORCL', 'V', 'MA',
    'XOM', 'PLTR', 'NFLX', 'JNJ', 'AMD', 'COST', 'BAC', 'ABBV',
    'HD', 'PG', 'GE', 'CVX', 'UNH', 'KO', 'CSCO', 'IBM',
    'WFC', 'CAT', 'MS', 'AXP', 'CRM', 'RTX', 'GS', 'TMUS',
    'PM', 'ABT', 'MRK', 'TMO', 'MCD', 'DIS', 'UBER', 'PEP',
    'LIN', 'QCOM', 'NOW', 'ISRG', 'INTC', 'C', 'INTU', 'T',
    'BLK', 'SCHW', 'NEE', 'VZ', 'BKNG', 'AMGN', 'ACN', 'BA',
    'DHR', 'GILD', 'TXN', 'ADBE', 'PFE', 'COF', 'LOW', 'UNP',
    'HON', 'DE', 'MDT', 'LMT', 'COP', 'SO', 'CMCSA', 'CVS',
    'DUK', 'NKE', 'MO', 'BMY', 'GD', 'SBUX', 'MMM', 'AMT',
    'UPS', 'EMR', 'BK', 'MDLZ', 'USB', 'PYPL', 'GM', 'CL',
    'FDX', 'SPG', 'MET', 'AIG'
]

# 합집합 + 지수 추가
all_tickers = list(set(nasdaq_100 + sp_100))
all_tickers.extend(['^IXIC', '^GSPC'])
all_tickers.sort()

print(f"✅ 총 종목 수: {len(all_tickers)}개")
print(f"✅ NASDAQ 100: {len(nasdaq_100)}개")
print(f"✅ S&P 100: {len(sp_100)}개")

## 6. Log Base 정규화 함수 정의

`log(P_t / P_0)` 공식으로 정규화 함수를 정의한다.

v1의 MinMaxScaler와의 핵심 차이: **MinMaxScaler를 적용하지 않고 로그값 그대로** 사용한다. 이렇게 하면 학습 시와 실전 추론 시 동일한 함수를 적용할 수 있어 재현성이 보장된다. MinMaxScaler를 쓰면 각 패턴의 min/max가 달라져 실전에서 동일한 스케일로 정규화하기 어렵다.

---

## 7. 슬라이딩 윈도우 + Log 정규화

12주 슬라이딩 윈도우로 패턴을 생성하고 Log Base 정규화를 적용한다. 3/6/12개월 수익률 메타데이터도 함께 저장한다.

In [ ]:
print(f"🚀 데이터 수집 시작: {len(all_tickers)}개 종목")
print(f"기간: 2020-01-01 ~ {datetime.now().strftime('%Y-%m-%d')}")
print("="*60)

success_count = 0
fail_count = 0
failed_tickers = []

for i, ticker in enumerate(all_tickers, 1):
    try:
        print(f"[{i:3d}/{len(all_tickers)}] {ticker:8s} 다운로드 중...", end=" ")

        data = yf.download(
            ticker,
            start='2020-01-01',
            interval='1wk',
            progress=False,
            auto_adjust=True
        )

        if not data.empty:
            ohlc_data = data[['Open', 'High', 'Low', 'Close']].copy()
            ohlc_data.columns = ['Open', 'High', 'Low', 'Close']

            file_path = f"{RAW_PATH}/{ticker}.csv"
            ohlc_data.index.name = "Date"
            ohlc_data.to_csv(file_path)

            print(f"✅ ({len(ohlc_data):3d}주)")
            success_count += 1
        else:
            print(f"❌ 데이터 없음")
            fail_count += 1
            failed_tickers.append(ticker)

    except Exception as e:
        print(f"❌ 에러: {str(e)[:40]}")
        fail_count += 1
        failed_tickers.append(ticker)

print("\n" + "="*60)
print(f"🎉 수집 완료!")
print(f"✅ 성공: {success_count}개 / ❌ 실패: {fail_count}개")
print("="*60)

## 8. 캔들스틱 이미지 생성

Log 정규화된 OHLC 패턴을 224×224 그레이스케일 캔들스틱 이미지로 변환한다.

음수 값(하락)도 mplfinance가 상대적 위치로 처리하므로 이미지 생성에 문제가 없다. 대비 1.5배 강화를 적용하고 `.npy` 형식으로 Colab 로컬에 저장한다.

---

## 9. tar 압축 및 Google Drive 백업

로컬 `images_v2` 폴더를 tar 파일로 압축하고 메타데이터·섹터 정보와 함께 Google Drive에 백업한다. 다음 세션에서 전처리를 반복하지 않아도 된다.

In [ ]:
print("📊 Sector/Industry 정보 수집 중...")

ticker_info_list = []

for ticker in tqdm(all_tickers, desc="Sector 수집"):
    try:
        info = yf.Ticker(ticker).info
        ticker_info_list.append({
            'ticker': ticker,
            'sector': info.get('sector', 'Unknown'),
            'industry': info.get('industry', 'Unknown')
        })
        time.sleep(0.1)
    except:
        ticker_info_list.append({
            'ticker': ticker,
            'sector': 'Unknown',
            'industry': 'Unknown'
        })

ticker_info_df = pd.DataFrame(ticker_info_list)
ticker_info_df.to_csv(f'{PROCESSED_PATH}/ticker_info.csv', index=False)

print(f"✅ 저장 완료: {PROCESSED_PATH}/ticker_info.csv")

## 🔄 Part 6: 정규화 함수 정의 (Log Only ⭐)

### 왜 Log만 사용하나요?
1. **학습과 실전 일치**: MinMaxScaler는 패턴마다 min/max가 달라서 실전에서 재현 불가
2. **금융 표준**: Log Return은 금융에서 가장 많이 쓰는 정규화 방식
3. **음수/양수 의미**: 음수 = 하락, 양수 = 상승으로 명확
4. **실전 사용**: 사용자 입력도 `log(ohlc / ohlc[0])`로 동일하게 처리

In [ ]:
def normalize_log_base(ohlc_array):
    """
    Log Base 정규화 (금융 표준 방식)

    공식: log(P_t / P_0)

    특징:
    - 양수: 상승 패턴
    - 음수: 하락 패턴
    - 0: 시작점 (기준)
    - Clipping 없음 (코로나 급등락 보존)

    Args:
        ohlc_array: (12, 4) numpy array

    Returns:
        (12, 4) Log 정규화된 numpy array

    실전 사용:
        user_ohlc = [...]  # 사용자 입력 12주
        log_norm = normalize_log_base(user_ohlc)  # 동일하게 처리!
    """
    first_close = ohlc_array[0, 3]  # 첫 번째 Close
    log_normalized = np.log(ohlc_array / first_close)  # log(P_t / P_0)

    return log_normalized  # MinMaxScaler 없음!

# 테스트
test_pattern = np.array([
    [100.0, 105.0, 98.0, 102.0],   # 시작 (log = 0)
    [102.0, 110.0, 101.0, 108.0],  # +5.9% 상승 (log = 0.057)
    [108.0, 115.0, 107.0, 112.0]   # +9.8% 상승 (log = 0.093)
])

normalized = normalize_log_base(test_pattern)
print("="*60)
print("✅ Log Base 정규화 함수 정의 완료")
print("="*60)
print("원본 OHLC:")
print(test_pattern)
print("\nLog 정규화 후:")
print(normalized)
print(f"\n범위: [{normalized.min():.3f}, {normalized.max():.3f}]")
print(f"\n✅ 양수 = 상승, 음수 = 하락, 0 = 시작점")
print("="*60)

## 🔄 Part 7: 슬라이딩 윈도우 + 메타데이터 (Log 정규화)

12주 슬라이딩 윈도우로 각 종목 패턴을 잘라 Log-Base 정규화 후 .npy로 저장합니다. 3/6/12개월 수익률과 섹터 정보를 함께 메타데이터 CSV에 기록합니다.

In [ ]:
print("="*60)
print("🔄 패턴 생성 + 메타데이터 (Log Base 정규화)")
print("="*60)

WINDOW_SIZE = 12
ticker_info_dict = ticker_info_df.set_index('ticker').to_dict('index')
csv_files = sorted([f for f in os.listdir(RAW_PATH) if f.endswith('.csv')])

all_metadata = []
success_count = 0
total_patterns = 0

# 예시를 위해 AAPL만 먼저 출력
example_ticker = None

for csv_file in tqdm(csv_files, desc="패턴 생성"):
    try:
        ticker = csv_file.replace('.csv', '')
        df = pd.read_csv(f"{RAW_PATH}/{csv_file}", index_col=0)

        sector = ticker_info_dict.get(ticker, {}).get('sector', 'Unknown')
        industry = ticker_info_dict.get(ticker, {}).get('industry', 'Unknown')

        ohlc = df[['Open', 'High', 'Low', 'Close']].values
        patterns = []
        num_patterns = len(ohlc) - WINDOW_SIZE + 1

        for i in range(num_patterns):
            window = ohlc[i:i+WINDOW_SIZE]

            # Log Base 정규화 ⭐
            normalized_window = normalize_log_base(window)
            patterns.append(normalized_window)

            start_idx = i
            end_idx = i + WINDOW_SIZE - 1
            start_price = df.iloc[start_idx]['Close']
            end_price = df.iloc[end_idx]['Close']

            idx_3m = end_idx + 12
            idx_6m = end_idx + 24
            idx_12m = end_idx + 52

            return_3m = ((df.iloc[idx_3m]['Close'] - end_price) / end_price * 100) if idx_3m < len(df) else None
            return_6m = ((df.iloc[idx_6m]['Close'] - end_price) / end_price * 100) if idx_6m < len(df) else None
            return_1y = ((df.iloc[idx_12m]['Close'] - end_price) / end_price * 100) if idx_12m < len(df) else None

            all_metadata.append({
                'ticker': ticker,
                'pattern_id': i,
                'start_date': df.index[start_idx],
                'end_date': df.index[end_idx],
                'start_price': start_price,
                'end_price': end_price,
                'sector': sector,
                'industry': industry,
                'return_3m': return_3m,
                'return_6m': return_6m,
                'return_1y': return_1y
            })

        # Numpy 배열 저장
        patterns_array = np.array(patterns)
        np.save(f"{PROCESSED_PATH}/{ticker}.npy", patterns_array)

        # AAPL 예시 저장
        if ticker == 'AAPL' and example_ticker is None:
            example_ticker = {
                'ticker': ticker,
                'num_patterns': len(patterns),
                'num_weeks': len(ohlc),
                'pattern_0': patterns[0],
                'pattern_1': patterns[1]
            }

        success_count += 1
        total_patterns += len(patterns)

    except Exception as e:
        print(f"\n❌ {ticker} 처리 실패: {e}")

# 메타데이터 저장
metadata_df = pd.DataFrame(all_metadata)
metadata_df.to_csv(f"{PROCESSED_PATH}/metadata_all.csv", index=False)

print("\n" + "="*60)
print(f"✅ 성공: {success_count}/{len(csv_files)}개")
print(f"📊 총 패턴: {total_patterns:,}개")
print("="*60)

# 예시 출력
if example_ticker:
    print("\n" + "="*60)
    print(f"📝 예시: {example_ticker['ticker']} 종목")
    print("="*60)
    print(f"주차 수: {example_ticker['num_weeks']}주")
    print(f"생성된 패턴: {example_ticker['num_patterns']}개\n")
    print(f"패턴 0 (정규화 후):")
    print(example_ticker['pattern_0'])
    print(f"\n범위: [{example_ticker['pattern_0'].min():.3f}, {example_ticker['pattern_0'].max():.3f}]")
    print("="*60)

## 🔄 Part 8: 이미지 생성 (그레이스케일 → .npy 저장)

### 저장 형식: .npy ⭐
- **PNG가 아닌 NumPy 배열 (.npy)로 저장**
- 이유:
  1. 빠른 로딩 (PNG 대비 10배 빠름)
  2. 압축 효율
  3. NumPy → Tensor 변환 쉬움
- 형식: (224, 224) uint8 배열

In [ ]:
def ohlc_to_grayscale_image(ohlc_array, temp_path='/content/temp_chart.png'):
    """
    OHLC → 224x224 그레이스케일 이미지

    Note: 음수 값도 mplfinance가 상대적 위치로 처리
    """
    try:
        df = pd.DataFrame(ohlc_array, columns=['Open', 'High', 'Low', 'Close'])
        df.index = pd.date_range('2020-01-01', periods=len(df), freq='W')

        mc = mpf.make_marketcolors(
            up='white', down='white',
            edge='white', wick='white',
            volume='white'
        )

        s = mpf.make_mpf_style(
            marketcolors=mc,
            gridstyle='',
            y_on_right=False,
            facecolor='black',
            figcolor='black'
        )

        mpf.plot(
            df, type='candle', style=s,
            savefig=temp_path,
            figsize=(2.24, 2.24),
            axisoff=True,
            closefig=True
        )

        img = Image.open(temp_path).convert('L')
        img_resized = img.resize((224, 224), Image.LANCZOS)
        enhancer = ImageEnhance.Contrast(img_resized)
        img_enhanced = enhancer.enhance(1.5)

        img_array = np.array(img_enhanced, dtype=np.uint8)

        img.close()
        img_resized.close()
        del img, img_resized, img_enhanced, enhancer, df

        if os.path.exists(temp_path):
            os.remove(temp_path)

        return img_array

    except Exception as e:
        return None

print("🚀 이미지 변환 시작 (코랩 저장소!)\n")

npy_files = sorted([f for f in os.listdir(PROCESSED_PATH) if f.endswith('.npy')])
total_success = 0
total_fail = 0

# 예시 저장
example_images = []

with tqdm(total=len(npy_files), desc="🔄 이미지 생성", unit="ticker") as pbar:
    for npy_file in npy_files:
        ticker = npy_file.replace('.npy', '')
        npy_path = os.path.join(PROCESSED_PATH, npy_file)
        patterns = np.load(npy_path)

        for i in range(len(patterns)):
            output_filename = f"{ticker}_{i}.npy"
            output_path = os.path.join(IMAGE_DIR, output_filename)

            pattern = patterns[i]
            img_array = ohlc_to_grayscale_image(pattern)

            if img_array is not None:
                np.save(output_path, img_array)  # .npy 형식으로 저장!
                total_success += 1

                # AAPL 첫 3개 예시 저장
                if ticker == 'AAPL' and i < 3:
                    example_images.append({
                        'filename': output_filename,
                        'shape': img_array.shape,
                        'dtype': img_array.dtype,
                        'range': (img_array.min(), img_array.max())
                    })
            else:
                total_fail += 1

            if (total_success + total_fail) % 100 == 0:
                gc.collect()

        del patterns
        gc.collect()

        pbar.update(1)
        pbar.set_postfix({'성공': total_success, '실패': total_fail})

print("\n" + "="*60)
print("🎉 이미지 변환 완료!")
print("="*60)
print(f"✅ 성공: {total_success:,}개")
print(f"❌ 실패: {total_fail}개")
print(f"💾 저장 위치: {IMAGE_DIR} (코랩 저장소)")
print(f"📦 저장 형식: .npy (NumPy 배열)")
print("="*60)

# 예시 출력
if example_images:
    print("\n📝 예시: AAPL 패턴 이미지")
    print("="*60)
    for ex in example_images:
        print(f"파일: {ex['filename']}")
        print(f"  Shape: {ex['shape']}")
        print(f"  Dtype: {ex['dtype']}")
        print(f"  Range: {ex['range'][0]} ~ {ex['range'][1]}")
        print()
    print("="*60)

## 📦 Part 9: tar 압축 및 Google Drive 백업

로컬 `images_v2` 폴더를 tar 파일로 압축하고 Google Drive로 복사합니다. 메타데이터 CSV와 섹터 정보도 함께 백업해 다음 세션에서 전처리를 반복하지 않아도 됩니다.

In [ ]:
# tar 압축
print("📦 tar 압축 중...")
!rm -f /content/data_v2/images_v2.tar
!tar -cf /content/data_v2/images_v2.tar -C /content/data_v2 images_v2/
!ls -lh /content/data_v2/images_v2.tar

# Google Drive로 복사
print("\n☁️ Google Drive로 백업 중...")
!cp /content/data_v2/images_v2.tar {GDRIVE_DATA_V2}/
!cp {PROCESSED_PATH}/metadata_all.csv {GDRIVE_DATA_V2}/
!cp {PROCESSED_PATH}/ticker_info.csv {GDRIVE_DATA_V2}/

# 확인
print("\n✅ 백업 완료!")
!ls -lh {GDRIVE_DATA_V2}/

## 🎉 Part 10: 완료!

### 저장된 파일
```
/content/drive/MyDrive/Patron/data_v2/
├── images_v2.tar (2.4GB)
├── metadata_all.csv
└── ticker_info.csv
```

### 다음 단계
1. 이 노트북을 맥북 로컬에 다운로드
2. patron_08_학습_v2.ipynb로 학습 진행
3. 구글드라이브에서 images_v2.tar 다운로드 → 로컬 저장
4. 코랩에서 로컬 파일 업로드 후 학습

**모든 전처리 완료! 🔥**